# Rückwärtskinematik des vereinfachten Scara Roboters durch Auflösen der Vorwärtskinematik nach den Gelenkwinkeln

Jonas Frei, 21.09.2026, jonas.frei@ost.ch

In [ ]:
import sympy as sp
from IPython.display import display

sp.init_printing()

## Auflösen der Vorwärtskinematik nach den Gelenkwinkeln

Die Vorwärtskinematik des vereinfachten Scara Roboters (zwei Drehgelenke, Längen $l_1$ und $l_2$) lautet:

$$x = l_1 \cos(q_1) + l_2 \cos(q_1+q_2)$$
$$y = l_1 \sin(q_1) + l_2 \sin(q_1+q_2)$$

Gesucht sind $q_1$ und $q_2$ in Abhängigkeit von $x$, $y$, $l_1$ und $l_2$.

In [ ]:
x, y, l1, l2, q1, q2 = sp.symbols('x y l1 l2 q1 q2')

eqns = [sp.Eq(x, l1*sp.cos(q1) + l2*sp.cos(q1 + q2)),
        sp.Eq(y, l1*sp.sin(q1) + l2*sp.sin(q1 + q2))]
display(eqns)

### Direkter Lösungsversuch

Der naheliegendste Ansatz ist, das Gleichungssystem direkt nach $q_1$ und $q_2$ aufzulösen, so wie man es von einem linearen Gleichungssystem gewohnt ist. Ein allgemeiner symbolischer Solver muss dazu mit den gekoppelten, **transzendenten** Gleichungen (Sinus und Kosinus von Summen der Unbekannten) zurechtkommen.

Sollte der nachfolgende Aufruf nach maximal 20 Sekunden nicht zu einem Resultat kommen oder mit einem Fehler abbrechen, findet der Solver warscheinlich keine Lösungen mehr. Brechen Sie ihn daher manuell ab. 

In [ ]:
sp.solve(eqns, [q1, q2], dict=True)

Ein allgemeiner symbolischer Solver findet für dieses gekoppelte transzendente Gleichungssystem also **innert nützlicher Zeit keine Lösung** -- die direkte Auflösung nach den Gelenkwinkeln ist nicht einfach möglich.

### Lösung über Hilfsvariablen

Ein gängiger Trick, um aus den transzendenten Gleichungen ein **algebraisches** (polynomiales) Gleichungssystem zu machen: Man führt $c_1=\cos(q_1)$, $s_1=\sin(q_1)$, $c_2=\cos(q_2)$, $s_2=\sin(q_2)$ als neue Unbekannte ein und ergänzt die beiden trigonometrischen Identitäten $c_1^2+s_1^2=1$ und $c_2^2+s_2^2=1$ als zusätzliche Gleichungen. Damit hat man vier Gleichungen für vier Unbekannte, die sich rein algebraisch lösen lassen.

In [ ]:
c1, s1, c2, s2 = sp.symbols('c1 s1 c2 s2')

poly_eqns = [
    sp.Eq(x, l1*c1 + l2*(c1*c2 - s1*s2)),
    sp.Eq(y, l1*s1 + l2*(s1*c2 + c1*s2)),
    sp.Eq(c1**2 + s1**2, 1),
    sp.Eq(c2**2 + s2**2, 1),
]

sol_trig = sp.solve(poly_eqns, [c1, s1, c2, s2], dict=True)
display(sol_trig)

Der algebraische Umweg **funktioniert** -- es gibt zwei Lösungen (die beiden Ellbogenkonfigurationen des Roboters) -- aber das Resultat ist alles andere als schön: lange, verschachtelte Ausdrücke mit Wurzeltermen für $\sin$ und $\cos$ der Gelenkwinkel, statt der Winkel selbst. Um daraus $q_1$ und $q_2$ zu erhalten, braucht es zusätzlich noch `atan2(sin, cos)`, was die Ausdrücke nicht wirklich übersichtlicher macht:

In [ ]:
q_sol = [
    [sp.atan2(sol[s1], sol[c1]), sp.atan2(sol[s2], sol[c2])] for sol in sol_trig
]

display(q_sol)

Das zeigt: Auch der "funktionierende" Weg liefert keine kompakte, direkt interpretierbare analytische Lösung.

## Auswertung

Wir werten die (unschöne) Lösung für konkrete Zahlenwerte aus und berechnen daraus die Gelenkwinkel.

In [ ]:
l1_val, l2_val = 1, 1
x_val, y_val = 1, 1
subs_dict = {l1: l1_val, l2: l2_val, x: x_val, y: y_val}

for q in q_sol:
    q_n = [sp.N(qi.subs(subs_dict)) for qi in q]
    print(f"q1 = {q_n[0]}   q2 = {q_n[1]}")

Zur Kontrolle setzen wir die berechneten Winkel wieder in die ursprüngliche Vorwärtskinematik ein: Beide Lösungen müssen $x=1$ und $y=1$ ergeben (bzw. den Wert, welcher für x_val und y_val vorgegeben wurde).

In [ ]:
for q in q_sol:
    q_n = [sp.N(qi.subs(subs_dict)) for qi in q]

    x_check = l1_val*sp.cos(q_n[0]) + l2_val*sp.cos(q_n[0] + q_n[1])
    y_check = l1_val*sp.sin(q_n[0]) + l2_val*sp.sin(q_n[0] + q_n[1])
    print(f"q1={q_n[0]}, q2={q_n[1]}  ->  x={sp.N(x_check)}, y={sp.N(y_check)}")